# Mi primer bot de telegram con python

## import de librerias

In [1]:
import telebot
from dotenv import load_dotenv
import os
import time
import random
import json
from telebot import types
from telebot.types import InlineKeyboardMarkup
from telebot.types import InlineKeyboardButton
from telebot.types import ForceReply
from comandos.common import ErrorUsuario
from comandos.estrategia import responder_estrategia, manejar_callback
from comandos.quepiloto import responder_quepiloto
import comandos.estrategia as e

## Instanciar el bot

In [2]:
load_dotenv()
tkn= os.getenv("TELEGRAM_TOKEN")
bot_inicio= telebot.TeleBot(tkn)

## Funciones de comandos
hay dos tipos de comandos: 
                   -los que empiezan con "/"
                   -cada comando tipo de mensaje :{texto,imagen,photo(cuando la sacás con la cámara), voice}


In [3]:
@bot_inicio.message_handler(commands=["text"])
def responder_html(mensaje):
    #Primera rta a enviar: html
        rta_html=""
    #Con etiquetas concatenadas:
        rta_html+="<b><u>Título formato HTML </u>:</b>"+"\n"
    #Forma de generar una rta con formato html
        rta_html += "<b> Texto en negrita </b>" +"\n"
        rta_html += "<i> Texto en cursiva </i>" +"\n"
        rta_html += "<u> Texto subrayado </u>" +"\n"
        rta_html += "<s> Texto tachado </s>" +"\n"
        rta_html += "<code> Monoespaciado </code>" +"\n"
        rta_html += '<a href="http://www.formula1.com/"> Link a web </a>' +"\n"
        rta_html += "<span class= 'tg-spoiler'> Texto spoiler </span>"
@bot_inicio.message_handler(commands= ["text"])
def responder_mark(mensaje):    
    #Segunda respuesta a enviar: formato markdown
        rta_mark=""
    #Con etiquetas concatenadas:
        rta_mark += "*__Título formato Markdown__:*"+"\n"
        rta_mark += "_Texto cursiva_:"+"\n"
        rta_mark += "__Text subrayado__:"+"\n"
        rta_mark += "~Texto tachado~"+"\n"
        rta_mark += "```Monoespacio```"+"\n"
        rta_mark += '[link a web] ("http://www.formula1.com/")'+"\n"
        rta_mark += "||Texto spoiler||"

In [4]:

@bot_inicio.message_handler(commands=["start"])
def cmd_start(message):
    print("me enviaron /start")
    bot_inicio.reply_to(message,"Hola, soy un bot que te puedo ayudar con la fórmula 1")

@bot_inicio.message_handler(commands=["saludar"])
def cmd_saludar(message):
    print("me enviaron /saludar")
    bot_inicio.reply_to(message,"Hola, soy un bot. Con que te gustaría comenzar?")



@bot_inicio.message_handler(commands=["ayuda"])
def ayuda(message):
     bot_inicio.reply_to(message,     "Soy un🏎️ Bot de F1\n\n"
                         "Que te puedo ayudar con:\n"
                         "/estrategia (Generemos una estrategia?) <año> <carrera> <piloto>\n   ej: /estrategia 2024 belgica HAM\n"
                         "/quepiloto (Con que piloto compartís el mes?) <mes>\n   ej: /quepiloto marzo\n\n"
                         "/historico (Aprendemos de historia?) <año> [carrera]\n   ej: /historico 2008  |  /historico 1994 brasil"
)

In [5]:
@bot_inicio.message_handler(commands="botones")
def cmd_botones(message):
    markup=InlineKeyboardMarkup(row_width=2)
    b1 = InlineKeyboardButton("Google", url="google.com.ar") 
    b2 = InlineKeyboardButton(" Página web - Fórmula 1", url="https://www.formula1.com/")
    b3 =InlineKeyboardButton("Instagram - Fórmula 1", url="https://www.instagram.com/f1/")
    b4=InlineKeyboardButton("Twitter - Fórmula 1", url="https://x.com/F1")
    markup.add(b1,b2,b3,b4)
    bot_inicio.send_message(message.chat.id, "Tus principales atajos son:", reply_markup=markup)

2026-10-28 12:27:16,287 (__init__.py:9465 MainThread) ERROR - TeleBot: "message_handler: Commands filter should be list of strings (commands), unknown type supplied to the 'commands' filter list. Not able to use the supplied type."


In [7]:
def crear_teclado(filas):
    teclado = InlineKeyboardMarkup()
    for fila in filas:
        botones = []
        for texto, dato in fila:
            boton = InlineKeyboardButton(
                texto,
                callback_data=dato
            )
            botones.append(boton)
        teclado.row(*botones)
    return teclado

@bot_inicio.message_handler(commands=["estrategia"])
def estrategia(message):
    args = message.text.split()[1:]
    try:
        texto, filas = responder_estrategia(args)
        if filas:
            teclado = crear_teclado(filas)
        else:
            teclado = None
        bot_inicio.send_message(
            message.chat.id,
            texto,
            reply_markup=teclado
        )
    except ErrorUsuario as error:
        bot_inicio.send_message(
            message.chat.id,
            str(error)
        )

@bot_inicio.callback_query_handler(
    func=lambda call: call.data.startswith("est:")
)
def responder_boton(call):
    try:
        texto, filas = manejar_callback(call.data)
        if filas:
            teclado = crear_teclado(filas)
        else:
            teclado = None
        bot_inicio.edit_message_text(
            texto,
            call.message.chat.id,
            call.message.message_id,
            reply_markup=teclado
        )
        bot_inicio.answer_callback_query(call.id)
    except ErrorUsuario as error:
        bot_inicio.answer_callback_query(
            call.id,
            str(error),
            show_alert=True
        )

In [9]:
@bot_inicio.message_handler(commands=["fin"])
def cmd_fin(message):
    bot_inicio.reply_to(message,"Hasta la proxima")

@bot_inicio.message_handler(commands=["problema"])
def cmd_problema(message):
    print("alguien tiene un problema")
    bot_inicio.reply_to(message,"Ok, ya se lo pase a un admin, en un rato será solucionado")
    print("se envio el problema del usuario")
    print(message)

    usuario = message.from_user.username
    nombre= message.from_user.first_name

    if usuario:
        identificador = f"@{usuario}"
    else:
        identificador = nombre

    msg = f"El usuario: {identificador}, (id: {message.chat.id}) tiene un problema"
    bot_inicio.send_message(8656118293,msg)
    bot_inicio.send_message(8797897882,msg)
    bot_inicio.send_message( 8695615052,msg)
@bot_inicio.message_handler(content_types=["text"])
def responder_mensaje(message):
    print("me enviaron un texto")
    if message.text.startswith("/"):
        bot_inicio.send_message(message.chat.id, "Comando inexistente")
    else:
         bot_inicio.send_message(message.chat.id, "me enviaste un texto, que todavia no puedo responder")


## Setear comandos
Acá se retiten los de acceso directo, de los 50 que tenés, se muestran 5 que son los más utilizados

In [10]:
bot_inicio.set_my_commands([
    telebot.types.BotCommand("/saludar", "saludar"),
    telebot.types.BotCommand("/problema", "ayuda de un admin"),
    telebot.types.BotCommand("/botones", "botones"),
    telebot.types.BotCommand("/fin","Finalizar")
])

True

## Lanzar el bot
Lanzamos el bot utilizando el while true
Y cuando estemos trabajando con el bot API, vamos a usar, infinity polling

In [11]:
if __name__== "__main__":
    print("Iniciando el bot")
    bot_inicio.infinity_polling()
    print("Fin")

Iniciando el bot
me enviaron /saludar


2026-10-28 12:28:25,103 (__init__.py:1160 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2026-10-28 12:28:25,105 (__init__.py:1162 MainThread) ERROR - TeleBot: "Break infinity polling"


Fin
